In [0]:
# We have orders , users and Products

In [0]:
from pyspark.sql import SparkSession
spark =SparkSession.builder\
    .appName("pyspark_optimization")\
    .getOrCreate()    

In [0]:
from pyspark.sql.functions import rand

In [0]:
orders = spark.range(0, 5_000_000).withColumnRenamed("id", "order_id") \
    .withColumn("user_id", (rand()*100000).cast("int")) \
    .withColumn("product_id", (rand()*1000).cast("int")) \
    .withColumn("category", (rand()*10).cast("int")) \
    .withColumn("amount", rand()*1000)

In [0]:
orders.show()

+--------+-------+----------+--------+------------------+
|order_id|user_id|product_id|category|            amount|
+--------+-------+----------+--------+------------------+
|       0|  74331|       969|       1| 470.0187559893966|
|       1|  69783|       733|       0| 625.9479754873898|
|       2|  60810|        16|       8| 846.7046264180926|
|       3|   8722|       461|       8|  356.366632094929|
|       4|  92522|       680|       6|13.911192041086151|
|       5|  20953|       607|       8| 410.8169118442292|
|       6|  53845|       245|       0| 393.3355968204362|
|       7|  44880|       607|       9|  314.063116471794|
|       8|   1253|       958|       9|435.63618178465555|
|       9|  71244|        57|       0| 730.8173148724353|
|      10|  98766|       111|       7|444.15752207855496|
|      11|  96604|       994|       9| 185.9515365981611|
|      12|  11868|       292|       1| 740.5381060468407|
|      13|  28379|       160|       6|  470.743631049104|
|      14|  73

In [0]:
#wrtie  orders

orders.write.mode("overwrite").parquet("/Volumes/workspace/default/end_to_end_pipeline/orders")

In [0]:
users = spark.range(0, 100000).withColumnRenamed("id", "user_id") \
    .withColumn("country", (rand()*5).cast("int"))
 
products = spark.range(0, 1000).withColumnRenamed("id", "product_id") \
    .withColumn("price", rand()*500)
 

In [0]:
users.write.mode("overwrite").parquet("/Volumes/workspace/default/end_to_end_pipeline/users")

In [0]:
products.write.mode("overwrite").parquet("/Volumes/workspace/default/end_to_end_pipeline/products")

In [0]:
#Read the data

orders = spark.read.parquet("/Volumes/workspace/default/end_to_end_pipeline/orders/")
users = spark.read.parquet("/Volumes/workspace/default/end_to_end_pipeline/users/")
products = spark.read.parquet("/Volumes/workspace/default/end_to_end_pipeline/products/")

In [0]:
orders.show(5)

+--------+-------+----------+--------+------------------+
|order_id|user_id|product_id|category|            amount|
+--------+-------+----------+--------+------------------+
| 1875000|  91612|       719|       5|379.14255019971085|
| 1875001|  31219|       226|       0|  570.448205058922|
| 1875002|  41969|       583|       3| 154.7892348961616|
| 1875003|  13059|       938|       3| 871.5708121654604|
| 1875004|   9376|       129|       4|203.08134035231996|
+--------+-------+----------+--------+------------------+
only showing top 5 rows


In [0]:
# Transformation

#Step 1. cleaning the category


In [0]:
from pyspark.sql.functions import col, lower, trim, broadcast, sum as _sum

In [0]:
orders_clean1 = orders.withColumn(
    "category",
    col("category").cast("string")
)

In [0]:
orders_clean = orders_clean1.withColumn(
    "category",
    lower(trim(col("category")))
)

In [0]:
#re_partition

orders_clean = orders_clean.repartition("user_id")
# it will redistribute the data accross the cluster based on user id so that processing is more balanced
# repartition is wide transformation

In [0]:
#optimizing joins
#using broadcast

df = orders_clean.join(broadcast(users),"user_id")\
    .join(products,"product_id")


caching

df.cache()

In [0]:
# find top product & revenue by country 

top_product  = df.groupby("product_id")\
    .count()\
    .orderBy(col("count").desc())\
    .limit(10)

# this is top n aggregation     

In [0]:
top_product.show(10)

+----------+-----+
|product_id|count|
+----------+-----+
|       285| 5225|
|       682| 5202|
|       876| 5198|
|       780| 5184|
|       231| 5179|
|       701| 5177|
|       943| 5173|
|       350| 5172|
|       559| 5171|
|       238| 5171|
+----------+-----+



In [0]:
# revenue per country

revenue = df.groupBy("country")\
    .agg(_sum("amount").alias("total_revenuw"))

In [0]:
# Wrtie data

top_products.write.mode("overwrite").parquet("/Volumes/workspace/default/end_to_end_pipeline/top_products")